# Laboratorio L11 - Un classificatore antiphishing

B.4 - Cybersicurezza e IA

Si addestra un piccolo classificatore di messaggi (phishing o legittimo) su un insieme di messaggi fittizi in italiano, e si osserva che cosa impara e come sbaglia. Il funzionamento interno del modello è approfondito nel corso B.6.

Il file `L11_messaggi.csv` deve trovarsi nella stessa cartella del notebook.

## 1. I dati

- `pd.read_csv`: legge il file in una tabella
- `tabella["etichetta"].value_counts()`: quanti messaggi per ciascuna etichetta

In [ ]:
import pandas as pd

dati = pd.read_csv("L11_messaggi.csv")
print(dati["etichetta"].value_counts())
dati.sample(5, random_state=1)

## 2. Addestramento e verifica (esercizio 1)

- `train_test_split`: divide i dati in insieme di addestramento (75%) e di verifica (25%); `random_state` rende la divisione ripetibile; `stratify` mantiene la stessa proporzione di etichette nei due insiemi
- `CountVectorizer`: trasforma ogni testo nei conteggi delle parole (sacchetto di parole)
- `MultinomialNB`: classificatore bayesiano ingenuo
- `fit` addestra, `predict` prevede
- `accuracy_score`: frazione di previsioni corrette
- `confusion_matrix`: tabella dei quattro casi (veri positivi, falsi positivi, falsi negativi, veri negativi)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    dati["testo"], dati["etichetta"], test_size=0.25, random_state=42, stratify=dati["etichetta"])

vettorizzatore = CountVectorizer()
modello = MultinomialNB()
modello.fit(vettorizzatore.fit_transform(X_train), y_train)

previsti = modello.predict(vettorizzatore.transform(X_test))
print("messaggi di addestramento:", len(X_train), " di verifica:", len(X_test))
print("accuratezza:", round(accuracy_score(y_test, previsti), 3))

etichette = ["phishing", "legittimo"]
cm = pd.DataFrame(confusion_matrix(y_test, previsti, labels=etichette),
                  index=["vero " + e for e in etichette], columns=["previsto " + e for e in etichette])
cm

**Nota**: i messaggi del dataset sono costruiti combinando un numero limitato di frasi, quindi molti si somigliano: l'accuratezza risulta più alta di quella che si otterrebbe con messaggi reali, molto più vari.

## 3. Che cosa ha imparato il modello (esercizio 2)

Il modello conserva, per ogni parola, una stima di quanto è frequente in ciascuna classe (`feature_log_prob_`, in scala logaritmica). La differenza tra le due classi indica quanto una parola spinge verso "phishing" o verso "legittimo".

- `get_feature_names_out()`: il vocabolario, cioè le parole conosciute dal modello
- `argsort()`: le posizioni che ordinano i valori dal più piccolo al più grande

In [ ]:
import numpy as np

parole = vettorizzatore.get_feature_names_out()
classi = list(modello.classes_)
diff = modello.feature_log_prob_[classi.index("phishing")] - modello.feature_log_prob_[classi.index("legittimo")]

ordine = diff.argsort()
print("parole più indicative di PHISHING :", ", ".join(parole[ordine[-15:][::-1]]))
print()
print("parole più indicative di LEGITTIMO:", ", ".join(parole[ordine[:15]]))

**Domande**: quali parole coincidono con le regole scritte a mano in L9? Quali sono nuove? Ce ne sono di sorprendenti (parole che non dovrebbero contare)?

Risposte:

## 4. Classificare nuovi messaggi (esercizio 3)

`predict_proba` restituisce la probabilità di ciascuna classe.

In [ ]:
def classifica(testo):
    p = modello.predict_proba(vettorizzatore.transform([testo]))[0]
    prob_phishing = p[classi.index("phishing")]
    return prob_phishing

NUOVI = [
    "SpediVeloce: il tuo pacco è in giacenza per indirizzo incompleto. Per la riconsegna paga 1,99 EUR di spese entro oggi",
    "Nella bacheca del registro elettronico è pubblicata la circolare n. 42 relativa all'uscita didattica",
    "Vi informo che i pagamenti delle fatture di settembre dovranno essere effettuati sul nuovo IBAN, procedete entro venerdì con riservatezza",
    "Ciao, ti ricordo che domani c'è la verifica di matematica, porta la calcolatrice",
    # aggiungere qui altri messaggi inventati
]
for t in NUOVI:
    print(f"P(phishing) = {classifica(t):.2f}   {t[:80]}")

## 5. Aggirare il classificatore (esercizio 4)

Riscrivere il messaggio fraudolento nella variabile `originale`, mantenendo la stessa richiesta (pagare le spese per un pacco), finché la probabilità di phishing scende sotto 0,5. Annotare le modifiche.

In [ ]:
originale = "Gentile cliente, SpediVeloce: il tuo pacco è in giacenza, paga le spese di 1,99 euro entro 24 ore al link"
riscritto = "Ciao! Grazie per il tuo acquisto: per la consegna di domani manca solo il contributo di 1,99 euro, trovi la ricevuta nella tua area personale"

print(f"originale : {classifica(originale):.2f}")
print(f"riscritto : {classifica(riscritto):.2f}")

**Domande**

- Quali modifiche hanno abbassato la probabilità?
- Una persona che applica la regola della verifica indipendente (L8) sarebbe stata ingannata dal messaggio riscritto? Perché?

Risposte:

## 6. La soglia (esercizio 5)

Il modello dice "phishing" se la probabilità supera 0,5. Cambiando la soglia cambiano i falsi positivi e i falsi negativi.

In [ ]:
prob_test = modello.predict_proba(vettorizzatore.transform(X_test))[:, classi.index("phishing")]
veri = (y_test == "phishing").to_numpy()

for soglia in [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    previsto_phishing = prob_test >= soglia
    falsi_positivi = int((previsto_phishing & ~veri).sum())
    falsi_negativi = int((~previsto_phishing & veri).sum())
    print(f"soglia {soglia:4}: falsi positivi {falsi_positivi:2}   falsi negativi {falsi_negativi:2}")

**Domanda**: quale soglia scegliereste per la posta della segreteria di una scuola? Che cosa costa di più, un falso positivo o un falso negativo?

Risposta: